In [8]:
from pydub import AudioSegment
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pyloudnorm as pyln

In [9]:
# --- Paths ---
INPUT_DIR  = Path(r"C:\Users\Dhanuja\Downloads\dataset\preprocessed audio")
OUTPUT_DIR = Path(r"C:\Users\Dhanuja\Downloads\dataset\chunked_data")

# --- Audio ---
TARGET_SR = 44100

# --- Chunking ---
CHUNK_MS     = 4000
OVERLAP_MS   = 2000
STEP_MS      = CHUNK_MS - OVERLAP_MS
MIN_TRACK_MS = 10_000

# --- Loudness ---
TARGET_LUFS     = -16.0
LUFS_CEILING    = -1.0
LUFS_FLOOR      = -28.0   # tracks quieter than this get skipped

In [10]:
def load_and_validate(filepath: Path) -> AudioSegment | None:
    try:
        audio = AudioSegment.from_file(filepath)
    except Exception as e:
        print(f"[SKIP] Could not load {filepath.name}: {e}")
        return None

    if len(audio) < MIN_TRACK_MS:
        print(f"[SKIP] Too short ({len(audio)/1000:.1f}s): {filepath.name}")
        return None

    # Convert to mono if not already
    if audio.channels != 1:
        audio = audio.set_channels(1)

    return audio

In [11]:
def normalize_lufs(audio: AudioSegment) -> AudioSegment | None:
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32) / 32768.0

    meter = pyln.Meter(TARGET_SR)
    loudness = meter.integrated_loudness(samples)

    if not np.isfinite(loudness):
        print("[WARN] Could not measure loudness, skipping")
        return None

    # Apply gain manually to avoid pyloudnorm internal clipping
    gain_db     = TARGET_LUFS - loudness
    gain_linear = 10 ** (gain_db / 20)
    normalized  = samples * gain_linear

    # True peak ceiling
    ceiling_linear = 10 ** (LUFS_CEILING / 20)
    peak = np.max(np.abs(normalized))
    if peak > ceiling_linear:
        normalized = normalized * (ceiling_linear / peak)

    pcm = (normalized * 32768.0).astype(np.int16)
    return AudioSegment(
        pcm.tobytes(),
        frame_rate=TARGET_SR,
        sample_width=2,
        channels=1
    )

In [12]:
def chunk_audio(audio: AudioSegment) -> list[tuple[int, AudioSegment]]:
    chunks = []
    start_ms = 0

    while start_ms + CHUNK_MS <= len(audio):
        chunk = audio[start_ms : start_ms + CHUNK_MS]
        chunks.append((start_ms, chunk))
        start_ms += STEP_MS

    return chunks

In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUPPORTED = {".mp3", ".wav", ".flac", ".ogg"}

files = [f for f in INPUT_DIR.rglob("*") if f.suffix.lower() in SUPPORTED]
print(f"Found {len(files)} tracks")

total_chunks = 0

for filepath in tqdm(files, desc="Processing"):
    audio = load_and_validate(filepath)
    if audio is None:
        continue

    audio = normalize_lufs(audio)
    if audio is None:
        continue

    chunks = chunk_audio(audio)

    out_folder = OUTPUT_DIR / filepath.stem
    out_folder.mkdir(parents=True, exist_ok=True)

    for i, (start_ms, chunk) in enumerate(chunks):
        out_name = f"{filepath.stem}_chunk{i:03d}_{start_ms}ms.mp3"
        chunk.export(out_folder / out_name, format="mp3", bitrate="320k")
        total_chunks += 1

print(f"\nDone. {total_chunks} chunks saved.")

Found 2036 tracks


Processing: 100%|██████████| 2036/2036 [1:03:08<00:00,  1.86s/it]


Done. 28502 chunks saved.


In [14]:
all_chunks = list(OUTPUT_DIR.rglob("*.mp3"))
print(f"Total chunks on disk: {len(all_chunks)}")
print(f"Tracks processed: {len(files)} / {len(files)}")

# Spot check one chunk
test = AudioSegment.from_file(all_chunks[0])
print(f"\nSample chunk:")
print(f"  Duration : {len(test)/1000}s")
print(f"  SR       : {test.frame_rate}")
print(f"  Channels : {test.channels}")

# Loudness distribution on 50 random chunks
import random
meter = pyln.Meter(TARGET_SR)
sample_chunks = random.sample(all_chunks, min(50, len(all_chunks)))
lufs_vals = []

for c in sample_chunks:
    a = AudioSegment.from_file(c)
    s = np.array(a.get_array_of_samples(), dtype=np.float32) / 32768.0
    l = meter.integrated_loudness(s)
    if np.isfinite(l):
        lufs_vals.append(l)

print(f"\nChunk loudness distribution (n={len(lufs_vals)}):")
print(f"  Mean : {np.mean(lufs_vals):.1f} LUFS")
print(f"  Std  : {np.std(lufs_vals):.1f} LU")
print(f"  Min  : {np.min(lufs_vals):.1f} LUFS")
print(f"  Max  : {np.max(lufs_vals):.1f} LUFS")

Total chunks on disk: 28502
Tracks processed: 2036 / 2036

Sample chunk:
  Duration : 4.0s
  SR       : 44100
  Channels : 1

Chunk loudness distribution (n=50):
  Mean : -17.0 LUFS
  Std  : 2.5 LU
  Min  : -31.3 LUFS
  Max  : -14.4 LUFS


In [15]:
# import random

# # Pick one track randomly or set manually
# test_file = Path(r"C:\Users\Dhanuja\Downloads\dataset\instrumentals\00000_Clockcleaner_When_My_Ship_Comes_In_instrumental.mp3")
# #test_file = random.choice([f for f in INPUT_DIR.rglob("*") if f.suffix.lower() in SUPPORTED])
# print(f"Testing on: {test_file.name}")
# print("-" * 60)

# # --- Step 1: Load ---
# raw = load_and_validate(test_file)
# if raw is None:
#     print("FAILED at load/validate")
# else:
#     print(f"[LOAD]     OK | duration: {len(raw)/1000:.1f}s | sr: {raw.frame_rate} | channels: {raw.channels}")

# # --- Step 2: Measure loudness BEFORE norm ---
# meter = pyln.Meter(TARGET_SR)
# samples_before = np.array(raw.get_array_of_samples(), dtype=np.float32) / 32768.0
# lufs_before = meter.integrated_loudness(samples_before)
# peak_before = np.max(np.abs(samples_before))
# print(f"[PRE-NORM] Loudness: {lufs_before:.1f} LUFS | Peak: {20*np.log10(peak_before):.1f} dBFS")

# # --- Step 3: Normalize ---
# normed = normalize_lufs(raw)
# if normed is None:
#     print("FAILED at normalization — track likely below LUFS floor")
# else:
#     samples_after = np.array(normed.get_array_of_samples(), dtype=np.float32) / 32768.0
#     lufs_after = meter.integrated_loudness(samples_after)
#     peak_after = np.max(np.abs(samples_after))
#     print(f"[POST-NORM] Loudness: {lufs_after:.1f} LUFS | Peak: {20*np.log10(peak_after):.1f} dBFS")

#     # Check normalization hit target
#     lufs_diff = abs(lufs_after - TARGET_LUFS)
#     print(f"[CHECK]    Target was {TARGET_LUFS} LUFS | Diff: {lufs_diff:.2f} LU", 
#           "✓" if lufs_diff < 1.0 else "✗ — exceeded 1 LU tolerance")
    
#     # Check ceiling wasn't breached
#     ceiling_linear = 10 ** (LUFS_CEILING / 20)
#     print(f"[CHECK]    True peak ceiling {LUFS_CEILING} dBFS:", 
#           "✓" if peak_after <= ceiling_linear else f"✗ — peak is {20*np.log10(peak_after):.1f} dBFS")

# # --- Step 4: Chunk ---
# chunks = chunk_audio(normed)
# print(f"\n[CHUNKING] {len(chunks)} chunks from {len(normed)/1000:.1f}s track")
# print(f"           Expected: {int((len(normed) - CHUNK_MS) / STEP_MS) + 1} chunks")
# print(f"           Match: {'✓' if len(chunks) == int((len(normed) - CHUNK_MS) / STEP_MS) + 1 else '✗'}")

# # --- Step 5: Check each chunk ---
# bad_chunks = []
# for i, (start_ms, chunk) in enumerate(chunks):
#     if len(chunk) != CHUNK_MS:
#         bad_chunks.append((i, len(chunk)))

# print(f"[CHECK]    All chunks exactly {CHUNK_MS}ms: {'✓' if not bad_chunks else f'✗ — {bad_chunks}'}")

# # --- Step 6: Export one chunk and reload ---
# test_out = OUTPUT_DIR / f"_test_{test_file.stem}_chunk000.mp3"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# _, first_chunk = chunks[0]
# first_chunk.export(test_out, format="mp3", bitrate="320k")

# reloaded = AudioSegment.from_file(test_out)
# print(f"\n[EXPORT]   Wrote and reloaded chunk successfully")
# print(f"           Duration : {len(reloaded)/1000:.1f}s (expected {CHUNK_MS/1000:.1f}s)")
# print(f"           SR       : {reloaded.frame_rate} (expected {TARGET_SR})")
# print(f"           Channels : {reloaded.channels} (expected 1)")

# # Cleanup test file
# test_out.unlink()
# print(f"\n[DONE] Test passed — safe to run full pipeline")